In [2]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="qwen/qwen3.6-27b",groq_api_key=groq_api_key,)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.5', 'langchain': '1.3.15'}}, client=<groq.resources.chat.completions.Completions object at 0x0000020BCF9986E0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000020BCF999160>, model_name='qwen/qwen3.6-27b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [4]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

C:\Users\utyag\AppData\Local\Temp\ipykernel_19008\2274412368.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_message_histories import ChatMessageHistory
c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [5]:
config={"configurable":{"session_id":"chat1"}}

In [6]:
from langchain_core.messages import HumanMessage

response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Ujjwal and I am a Senior AI/ML Engineer")],
    config=config
)

In [7]:
print(response.content.split("</think>")[-1])



Hi Ujjwal! Nice to meet you. It's great to connect with a Senior AI/ML Engineer. 

How can I assist you today? Whether you're looking to brainstorm model architectures, optimize training pipelines, discuss MLOps/LLMOps workflows, debug performance bottlenecks, or dive into the latest research and tooling, I'm here to help. What's on your mind?


In [8]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)
print(response.content.split("</think>")[-1])




Your name is Ujjwal. How can I help you today?


In [9]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config1,
)
print(response.content.split("</think>")[-1])




I don't know your name yet! I don't have access to personal information unless you share it with me in our conversation. If you'd like, just tell me your name and I'll gladly use it going forward. 😊


In [10]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
print(response.content.split("</think>")[-1])



Nice to meet you, John! How can I help you today? 😊


In [11]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
print(response.content.split("</think>")[-1])



Your name is John! How can I help you today? 😊


### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [12]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the nest of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model
chain

ChatPromptTemplate(input_variables=['messages'], input_types={'messages': list[typing.Annotated[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')] | typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')] | typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')] | typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')] | typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')] | typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')] | typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')] | typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')] | typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')] | typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')] | typing.Annotated[langchain_core.messag

In [13]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Ujjwal")]})

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name is Ujjwal"\n   - This is a simple greeting and introduction.\n   - No specific question or request is made.\n\n2.  **Identify Key Elements:**\n   - Greeting: "Hi"\n   - Name: "Ujjwal"\n   - Intent: Introduction, likely expecting a friendly response and perhaps an offer to help.\n\n3.  **Determine Response Strategy:**\n   - Acknowledge the greeting\n   - Use the user\'s name to personalize the response\n   - Express willingness to help\n   - Keep it friendly and open-ended\n\n4.  **Draft Response (Mental):**\n   Hi Ujjwal! Nice to meet you. How can I assist you today? Feel free to ask me anything or let me know what\'s on your mind!\n\n5.  **Refine Response (Self-Correction/Verification):**\n   - Matches tone: Friendly, professional\n   - Uses name correctly: Yes\n   - Invites further interaction: Yes\n   - No overcomplication: Keeps it simple\n   - Ready.\n\n6.  **Outpu

In [14]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [15]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Krish")],
    config=config
)

response

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "Hi My name is Krish"\n   - This is a simple greeting and introduction.\n   - Key information: Name = Krish\n\n2.  **Identify Intent:**\n   - The user is introducing themselves.\n   - Expected response: Acknowledge the greeting, use their name, and offer assistance.\n\n3.  **Determine Response Strategy:**\n   - Greet back warmly\n   - Acknowledge their name (Krish)\n   - Ask how I can help them\n   - Keep it friendly and open-ended\n\n4.  **Draft Response (Mental):**\n   Hi Krish! It\'s nice to meet you. How can I assist you today?\n\n5.  **Refine Response:**\n   - Check tone: Friendly, professional\n   - Check accuracy: Uses correct name\n   - Check completeness: Invites next step\n   - Matches guidelines: Direct, helpful\n\n   Final: "Hi Krish! Nice to meet you. How can I help you today?" \n\n6.  **Self-Correction/Verification:**\n   - No factual claims to verify\n   - Tone is ap

In [16]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

print(response.content.split("</think>")[-1])



Your name is **Krish**! How can I help you today?


In [17]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [18]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Krish")],"language":"Hindi"})
print(response.content.split("</think>")[-1])



नमस्ते कृष्ण! आपका परिचय पढ़कर बहुत अच्छा लगा। बताइए, आज मैं आपकी कैसे मदद कर सकता हूँ? 😊


Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [19]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [20]:
config = {"configurable": {"session_id": "chat5"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Krish")],"language":"Hindi"},
    config=config
)
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - User says: "Hi My name is Krish"
   - Language: English
   - Intent: Greeting and introduction
   - Key information: Name is Krish

2.  **Identify Constraints:**
   - System prompt: "Answer all questions to the best of your ability in Hindi."
   - This means I must respond in Hindi, regardless of the input language.

3.  **Formulate Response (Internal Translation & Structuring):**
   - Acknowledge the greeting: Hello/Hi -> नमस्ते / हेलो
   - Acknowledge the name: My name is Krish -> आपका नाम कृष्ण है / आपका परिचय स्वीकार है
   - Respond politely in Hindi: नमस्ते कृष्ण! आपका परिचय पढ़कर अच्छा लगा। बताइए, मैं आपकी कैसे मदद कर सकता हूँ?
   - Check tone: Friendly, helpful, in Hindi as required.

4.  **Refine Response (Hindi):**
   नमस्ते कृष्ण! आपका परिचय पढ़कर बहुत अच्छा लगा। बताइए, आज मैं आपकी कैसे मदद कर सकता हूँ?

5.  **Self-Correction/Verification:**
   - Does it follow the constraint? Yes, entirely in Hindi.
   - D

In [21]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [22]:
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: "whats my name?"\n   - Previous context: User introduced themselves as "Krish" in the first message.\n   - Language: User is asking in English, but I\'m instructed to answer in Hindi.\n\n2.  **Identify Key Information:**\n   - User\'s name: Krish\n   - Need to respond in Hindi as per system instruction.\n\n3.  **Formulate Response (Internal Translation & Structuring):**\n   - English: "Your name is Krish."\n   - Hindi: "आपका नाम क्रिष है।"\n   - Add polite/contextual elements if needed: "जैसा कि आपने पहले बताया, आपका नाम क्रिष है।" (As you mentioned earlier, your name is Krish.)\n\n4.  **Check Constraints:**\n   - Must answer in Hindi.\n   - Accurate to context.\n   - Keep it natural and helpful.\n\n5.  **Final Output Generation:**\n   "आपका नाम क्रिष है। जैसा कि आपने शुरुआत में बताया था। 😊" (Your name is Krish, as you mentioned at the beginning.)\n\n   This matches the requirement. I\'ll output thi

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [24]:
from langchain_core.messages import SystemMessage, AIMessage, trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\langchain_core\language_models\base.py:448: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [25]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What ice cream do i like"\n2.  **Identify Key Information Needed:** To answer this, I need to know the user\'s personal preferences, specifically regarding ice cream.\n3.  **Check Available Context:** I have no prior context about the user\'s ice cream preferences. This is a fresh conversation with only a few basic exchanges (2+2, thanks, having fun?).\n4.  **Determine Limitations:** As an AI, I don\'t have access to personal information about users unless they explicitly share it in the current conversation or it\'s stored in a memory/profile system (which I don\'t have access to in this standard setup).\n5.  **Formulate Response:** \n   - Acknowledge the question\n   - Explain that I don\'t know their preferences\n   - Ask them to tell me what they like\n   - Keep it friendly and conversational\n   - Example: "I don’t actually know! I’d love to hear what your favorite is though. Are you team chocola

In [26]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "what math problem did i ask"\n   - I need to recall the previous conversation.\n   - Earlier, the user asked: "whats 2 + 2"\n\n2.  **Identify Key Information:**\n   - The math problem was "2 + 2"\n   - I answered "4"\n   - The user then said "thanks", I said "no problem!", user said "having fun?", I said "yes!"\n   - Now they\'re asking what math problem they asked.\n\n3.  **Formulate Response:**\n   - Keep it concise and accurate based on the conversation history.\n   - State clearly: "You asked what 2 + 2 equals." or "You asked \'whats 2 + 2\'."\n\n4.  **Check Constraints:**\n   - Language: English\n   - Tone: Helpful, accurate\n   - No extra fluff needed.\n\n5.  **Draft Response:**\n   You asked "whats 2 + 2". \n\n   (Self-Correction/Refinement during drafting: Make it slightly more natural but keep it direct.)\n   "You asked what 2 + 2 equals." or "You asked \'what\'s 2 + 2\'." Both work. I\'ll

In [27]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Users\utyag\Documents\Projects\Agentic-AI\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3775: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [28]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "whats my name?"\n2.  **Check Context/History:** I need to scan the conversation history for any mention of the user\'s name.\n   - User: "I like vanilla ice cream"\n   - Me: "nice"\n   - User: "whats 2 + 2"\n   - Me: "4"\n   - User: "thanks"\n   - Me: "no problem!"\n   - User: "having fun?"\n   - Me: "yes!"\n   - User: "whats my name?"\n3.  **Identify Missing Information:** The user has not provided their name in this conversation.\n4.  **Formulate Response:** I should politely state that I don\'t know their name because they haven\'t told me yet, and ask if they\'d like to share it. Keep it consistent with the conversational tone.\n5.  **Draft Response:** you haven\'t told me your name yet! what should i call you?\n6.  **Refine (Self-Correction/Verification):** Check tone: friendly, direct. Matches previous lowercase style? The user has been using lowercase, but I\'ve been using standard capitalizati

In [29]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User asks: "what math problem did i ask?"\n   - This implies they think they previously asked a math problem in this conversation.\n   - However, looking at the conversation history, this is the very first message. There is no prior context or previous math problem.\n\n2.  **Identify Key Issue:**\n   - No prior messages exist in this session.\n   - I need to politely inform the user that this is the first message in our conversation, so they haven\'t asked a math problem yet.\n   - I should also offer to help with any math problem they\'d like to ask.\n\n3.  **Formulate Response:**\n   - Acknowledge the question.\n   - Clarify that this is the start of our conversation.\n   - State that no math problem has been asked yet.\n   - Invite them to share the problem they\'d like help with.\n   - Keep it clear, polite, and in English as instructed.\n\n   Draft: "This is actually the first message in our conversation,